# GPM IMERG monthly precipitation (NASA Earthdata)

Download a monthly GPM IMERG precipitation granule from GES DISC through one Earthdata Login, then map it. This is a **live** query: it needs the `[earthdata]` extra (Python >=3.12) and EDL credentials (`EARTHDATA_USERNAME` / `EARTHDATA_PASSWORD` or `~/.netrc`). The query is wrapped so the notebook stays safe under nbval-lax when run offline / without credentials. See [Authentication](../../reference/earthdata/authentication.md) and [Usage](../../reference/earthdata/usage.md).

In [ ]:
from pathlib import Path

from earthlens import EarthLens

OUT_DIR = Path('earthdata_output')
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
paths = None
try:
    paths = EarthLens(
        data_source='earthdata',
        variables={'GPM_3IMERGM_07': ['precipitation']},
        start='2023-06-01',
        end='2023-06-30',
        lat_lim=[-60.0, 60.0],
        lon_lim=[-180.0, 180.0],
        path=str(OUT_DIR),
    ).download(progress_bar=False)
    print(len(paths), 'granule(s):', [Path(p).name for p in paths])
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

In [ ]:
# Open the fetched granule and map mean precipitation. Wrapped so a read /
# plotting surprise does not mask the (separately asserted) download above.
if paths:
    try:
        import matplotlib.pyplot as plt
        import xarray as xr

        ds = xr.open_dataset(paths[0], group='Grid')
        precip = ds['precipitation'].isel(time=0)
        fig, ax = plt.subplots(figsize=(10, 5))
        precip.where(precip >= 0).T.plot(ax=ax, cmap='viridis', robust=True)
        ax.set_title('GPM IMERG monthly precipitation rate, June 2023')
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f'plot step skipped: {type(exc).__name__}: {exc}')